# Acne Detection Training Pipeline (PyTorch)
This notebook trains a production-ready CNN classifier using your dataset folders as classes.

Pipeline features:
- class-aware train/val/test split,
- strong but safe augmentation,
- imbalance handling with weighted sampling and class-weighted loss,
- pretrained CNN fine-tuning,
- early stopping + LR scheduling,
- mixed precision training,
- best-model checkpointing and export for inference.

In [3]:
# If needed, uncomment this once:
# !pip install -q torch torchvision pandas numpy matplotlib tqdm pillow

import os
import json
import time
import copy
import math
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, transforms, models

In [5]:
@dataclass
class Config:
    dataset_root: str = "/home/dev/Desktop/cnn/dataset"
    output_dir: str = "/home/dev/Desktop/cnn/artifacts"
    run_name: str = "acne_efficientnet_b0"

    img_size: int = 224
    batch_size: int = 32
    num_workers: int = 4

    epochs: int = 35
    lr: float = 3e-4
    weight_decay: float = 1e-4
    label_smoothing: float = 0.05

    train_ratio: float = 0.75
    val_ratio: float = 0.15
    test_ratio: float = 0.10

    patience: int = 7
    min_delta: float = 1e-4
    freeze_backbone_epochs: int = 2

    use_weighted_sampler: bool = True
    use_class_weighted_loss: bool = True

    seed: int = 42

cfg = Config()
assert abs((cfg.train_ratio + cfg.val_ratio + cfg.test_ratio) - 1.0) < 1e-8, "Split ratios must sum to 1.0"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)

RUN_DIR = Path(cfg.output_dir) / cfg.run_name
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Run directory:", RUN_DIR)

Device: cpu
Run directory: /home/dev/Desktop/cnn/artifacts/acne_efficientnet_b0


In [6]:
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(cfg.img_size, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.20, scale=(0.01, 0.10), ratio=(0.3, 3.3), value='random')
])

eval_tfms = transforms.Compose([
    transforms.Resize(int(cfg.img_size * 1.15)),
    transforms.CenterCrop(cfg.img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

base_dataset = datasets.ImageFolder(cfg.dataset_root)
class_names = base_dataset.classes
num_classes = len(class_names)
print("Classes:", class_names)
print("Num classes:", num_classes)

# Save class map for inference
with open(RUN_DIR / "class_to_idx.json", "w") as f:
    json.dump(base_dataset.class_to_idx, f, indent=2)

Classes: ['Acne', 'Acne_Blackheads', 'Acne_Cystic', 'Acne_Pustules', 'Acne_Whiteheads', 'Carcinoma', 'Eczema', 'Keratosis', 'Normal', 'Rosacea', 'Vitiligo']
Num classes: 11


In [7]:
targets = np.array(base_dataset.targets)
indices = np.arange(len(base_dataset))

# Stratified split without external dependencies
by_class = defaultdict(list)
for idx, y in zip(indices, targets):
    by_class[int(y)].append(int(idx))

train_idx, val_idx, test_idx = [], [], []
for cls, cls_indices in by_class.items():
    cls_indices = np.array(cls_indices)
    rng = np.random.default_rng(cfg.seed + cls)
    rng.shuffle(cls_indices)

    n = len(cls_indices)

    if n == 1:
        n_train, n_val, n_test = 1, 0, 0
    elif n == 2:
        n_train, n_val, n_test = 1, 0, 1
    else:
        n_train = max(1, int(round(n * cfg.train_ratio)))
        n_val = max(1, int(round(n * cfg.val_ratio)))
        n_test = n - n_train - n_val

        if n_test < 1:
            deficit = 1 - n_test
            reducible_train = max(0, n_train - 1)
            take_train = min(deficit, reducible_train)
            n_train -= take_train
            deficit -= take_train

            reducible_val = max(0, n_val - 1)
            take_val = min(deficit, reducible_val)
            n_val -= take_val
            deficit -= take_val

            n_test = n - n_train - n_val

        if n_train + n_val + n_test != n:
            n_test = n - n_train - n_val

    train_idx.extend(cls_indices[:n_train].tolist())
    val_idx.extend(cls_indices[n_train:n_train + n_val].tolist())
    test_idx.extend(cls_indices[n_train + n_val:n_train + n_val + n_test].tolist())

if len(val_idx) == 0:
    val_idx.append(train_idx.pop())
if len(test_idx) == 0:
    test_idx.append(train_idx.pop())

print(f"Train/Val/Test sizes: {len(train_idx)}, {len(val_idx)}, {len(test_idx)}")

train_dataset_full = datasets.ImageFolder(cfg.dataset_root, transform=train_tfms)
val_dataset_full = datasets.ImageFolder(cfg.dataset_root, transform=eval_tfms)
test_dataset_full = datasets.ImageFolder(cfg.dataset_root, transform=eval_tfms)

train_subset = Subset(train_dataset_full, train_idx)
val_subset = Subset(val_dataset_full, val_idx)
test_subset = Subset(test_dataset_full, test_idx)

# Class distribution check
train_targets = targets[train_idx]
val_targets = targets[val_idx]
test_targets = targets[test_idx]

dist_df = pd.DataFrame({
    "class": class_names,
    "train": np.bincount(train_targets, minlength=num_classes),
    "val": np.bincount(val_targets, minlength=num_classes),
    "test": np.bincount(test_targets, minlength=num_classes),
})
display(dist_df)

Train/Val/Test sizes: 17474, 3494, 2331


,class,train,val,test
0,Acne,3759,752,501
1,Acne_Blackheads,1607,321,215
2,Acne_Cystic,250,50,34
3,Acne_Pustules,242,48,32
4,Acne_Whiteheads,72,14,10
5,Carcinoma,299,60,40
6,Eczema,970,194,129
7,Keratosis,1375,275,183
8,Normal,7610,1522,1015
9,Rosacea,774,155,103


In [5]:
# Imbalance handling: WeightedRandomSampler + class-weighted loss
class_counts_train = np.bincount(train_targets, minlength=num_classes)
class_weights = 1.0 / np.maximum(class_counts_train, 1)
class_weights = class_weights / class_weights.mean()

sample_weights = class_weights[train_targets]
train_sampler = None
train_shuffle = True
if cfg.use_weighted_sampler:
    train_sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )
    train_shuffle = False

train_loader = DataLoader(
    train_subset,
    batch_size=cfg.batch_size,
    shuffle=train_shuffle,
    sampler=train_sampler,
    num_workers=cfg.num_workers,
    pin_memory=(DEVICE == "cuda"),
)
val_loader = DataLoader(
    val_subset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=(DEVICE == "cuda"),
)
test_loader = DataLoader(
    test_subset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=(DEVICE == "cuda"),
)

print("Train class counts:", class_counts_train.tolist())
print("Class weights:", class_weights.round(3).tolist())

Train class counts: [3759, 1607, 250, 242, 72, 299, 970, 1375, 7610, 774, 516]
Class weights: [0.093, 0.218, 1.402, 1.449, 4.87, 1.173, 0.361, 0.255, 0.046, 0.453, 0.679]


In [6]:
weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
model = models.efficientnet_b0(weights=weights)

# Replace classification head
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)
model = model.to(DEVICE)

# Optional staged fine-tuning: freeze feature extractor first
for p in model.features.parameters():
    p.requires_grad = False

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE) if cfg.use_class_weighted_loss else None,
    label_smoothing=cfg.label_smoothing,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

print(model.classifier)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /home/dev/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:04<00:00, 4.94MB/s]

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=11, bias=True)
)



/tmp/ipykernel_108715/99954762.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))


In [7]:
def accuracy_from_logits(logits, labels):
    preds = logits.argmax(dim=1)
    return (preds == labels).float().mean().item()


def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    is_train = optimizer is not None
    model.train(is_train)

    running_loss = 0.0
    running_correct = 0
    total = 0

    y_true = []
    y_pred = []

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        if is_train:
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            scaler.step(optimizer)
            scaler.update()

        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == y).sum().item()
        total += x.size(0)

        y_true.append(y.detach().cpu())
        y_pred.append(preds.detach().cpu())

    epoch_loss = running_loss / max(total, 1)
    epoch_acc = running_correct / max(total, 1)

    y_true = torch.cat(y_true).numpy() if y_true else np.array([])
    y_pred = torch.cat(y_pred).numpy() if y_pred else np.array([])

    return epoch_loss, epoch_acc, y_true, y_pred


def per_class_accuracy(y_true, y_pred, num_classes):
    out = {}
    for c in range(num_classes):
        mask = (y_true == c)
        if mask.sum() == 0:
            out[c] = np.nan
        else:
            out[c] = float((y_pred[mask] == y_true[mask]).mean())
    return out

In [8]:
history = []
best_state = None
best_val_acc = -np.inf
patience_counter = 0

start = time.time()
for epoch in range(1, cfg.epochs + 1):
    # unfreeze backbone after warmup
    if epoch == cfg.freeze_backbone_epochs + 1:
        for p in model.features.parameters():
            p.requires_grad = True
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr * 0.3, weight_decay=cfg.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=2
        )
        print("Backbone unfrozen for full fine-tuning.")

    train_loss, train_acc, _, _ = run_epoch(model, train_loader, criterion, optimizer, scaler)
    val_loss, val_acc, yv, pv = run_epoch(model, val_loader, criterion, optimizer=None, scaler=None)

    scheduler.step(val_acc)

    val_pc = per_class_accuracy(yv, pv, num_classes)
    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "lr": optimizer.param_groups[0]["lr"],
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d}/{cfg.epochs} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | lr={row['lr']:.2e}"
    )

    improved = (val_acc - best_val_acc) > cfg.min_delta
    if improved:
        best_val_acc = val_acc
        best_state = {
            "model": copy.deepcopy(model.state_dict()),
            "epoch": epoch,
            "val_acc": val_acc,
            "config": asdict(cfg),
            "class_names": class_names,
            "class_to_idx": base_dataset.class_to_idx,
        }
        torch.save(best_state, RUN_DIR / "best_model.pt")
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= cfg.patience:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

elapsed = time.time() - start
print(f"Training completed in {elapsed/60:.1f} minutes. Best val acc: {best_val_acc:.4f}")

history_df = pd.DataFrame(history)
history_df.to_csv(RUN_DIR / "train_history.csv", index=False)
display(history_df.tail())

/tmp/ipykernel_108715/205353144.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):


Epoch 01/35 | train_loss=1.2141 train_acc=0.4249 | val_loss=3.8371 val_acc=0.2616 | lr=3.00e-04
Epoch 02/35 | train_loss=0.8202 train_acc=0.5675 | val_loss=3.6661 val_acc=0.4273 | lr=3.00e-04
Backbone unfrozen for full fine-tuning.
Epoch 03/35 | train_loss=0.4946 train_acc=0.7595 | val_loss=2.9292 val_acc=0.6365 | lr=9.00e-05
Epoch 04/35 | train_loss=0.3516 train_acc=0.8530 | val_loss=2.7804 val_acc=0.7078 | lr=9.00e-05
Epoch 05/35 | train_loss=0.3139 train_acc=0.8941 | val_loss=2.6479 val_acc=0.7785 | lr=9.00e-05
Epoch 06/35 | train_loss=0.3095 train_acc=0.9108 | val_loss=2.5906 val_acc=0.7965 | lr=9.00e-05
Epoch 07/35 | train_loss=0.2987 train_acc=0.9243 | val_loss=2.6050 val_acc=0.7696 | lr=9.00e-05
Epoch 08/35 | train_loss=0.2865 train_acc=0.9345 | val_loss=2.5050 val_acc=0.7928 | lr=9.00e-05
Epoch 09/35 | train_loss=0.2843 train_acc=0.9412 | val_loss=2.4249 val_acc=0.8397 | lr=9.00e-05
Epoch 10/35 | train_loss=0.2766 train_acc=0.9499 | val_loss=2.4370 val_acc=0.8515 | lr=9.00e-05


KeyboardInterrupt: 

In [8]:
# Load best checkpoint for final evaluation
ckpt = torch.load(RUN_DIR / "best_model.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()

test_loss, test_acc, yt, pt = run_epoch(model, test_loader, criterion, optimizer=None, scaler=None)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

pc_acc = per_class_accuracy(yt, pt, num_classes)
pc_df = pd.DataFrame({
    "class": class_names,
    "test_class_acc": [pc_acc[i] for i in range(num_classes)]
})
display(pc_df)
pc_df.to_csv(RUN_DIR / "test_per_class_accuracy.csv", index=False)

NameError: name 'model' is not defined

In [9]:
# Save production inference artifact
export = {
    "model_state_dict": model.state_dict(),
    "class_names": class_names,
    "class_to_idx": base_dataset.class_to_idx,
    "img_size": cfg.img_size,
    "normalize_mean": [0.485, 0.456, 0.406],
    "normalize_std": [0.229, 0.224, 0.225],
    "architecture": "efficientnet_b0",
}
torch.save(export, RUN_DIR / "model_inference.pt")

with open(RUN_DIR / "config.json", "w") as f:
    json.dump(asdict(cfg), f, indent=2)

print("Saved artifacts:")
for p in sorted(RUN_DIR.glob("*")):
    print("-", p.name)

NameError: name 'model' is not defined

In [10]:
# Optional: quick inference helper for a single image
inference_tfms = transforms.Compose([
    transforms.Resize(int(cfg.img_size * 1.15)),
    transforms.CenterCrop(cfg.img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def predict_image(image_path: str, checkpoint_path: str = None):
    checkpoint_path = checkpoint_path or str(RUN_DIR / "model_inference.pt")
    blob = torch.load(checkpoint_path, map_location=DEVICE)

    net = models.efficientnet_b0(weights=None)
    in_features = net.classifier[1].in_features
    net.classifier[1] = nn.Linear(in_features, len(blob["class_names"]))
    net.load_state_dict(blob["model_state_dict"])
    net = net.to(DEVICE).eval()

    with Image.open(image_path) as im:
        x = inference_tfms(im.convert("RGB")).unsqueeze(0).to(DEVICE)

    with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
        logits = net(x)
        probs = torch.softmax(logits, dim=1)[0]

    conf, idx = torch.max(probs, dim=0)
    pred_class = blob["class_names"][idx.item()]
    return {
        "pred_class": pred_class,
        "confidence": float(conf.item()),
        "top3": [
            (blob["class_names"][i], float(p))
            for i, p in sorted(enumerate(probs.cpu().numpy().tolist()), key=lambda z: z[1], reverse=True)[:3]
        ],
    }

# Example:
# predict_image('/home/dev/Desktop/cnn/dataset/Acne/sample.jpg')

In [15]:
predict_image('image.png')

/tmp/ipykernel_4856/2577776600.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):


{'pred_class': 'Acne_Whiteheads',
 'confidence': 0.31415826082229614,
 'top3': [('Acne_Whiteheads', 0.31415826082229614),
  ('Acne', 0.21449922025203705),
  ('Eczema', 0.18792225420475006)]}